# Factor Return Correlation Heatmap

Load each cached stock universe, align fundamentals to their first actionable trading dates, calculate every registered factor portfolio within its own cohort, concatenate the period returns, and compare the resulting longer factor-return series.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt

from modules.factors.mappings import FACTOR_MAPPINGS
from modules.redundancy.heatmap import create_correlation_heatmap
from notebooks.utils import (
    calculate_factor_return_data,
    concatenate_factor_return_periods,
    load_aligned_factor_data,
)

In [ ]:
PERIOD_YEARS = (2015, 2020)  # Add 2025 after that cache is ready.
list(FACTOR_MAPPINGS)

In [ ]:
aligned_periods = {}
for year in PERIOD_YEARS:
    period_data = load_aligned_factor_data(year=year)
    aligned_periods[year] = period_data
    print(
        f'{year}-{year + 4}: {period_data["ticker"].nunique():,} stocks; '
        f'{len(period_data):,} daily observations'
    )

In [ ]:
period_factor_returns = [
    calculate_factor_return_data(data=aligned_periods[year], year=year)
    for year in PERIOD_YEARS
]
factor_returns = concatenate_factor_return_periods(period_factor_returns)
print(
    f'{factor_returns["date"].min():%Y-%m-%d} to '
    f'{factor_returns["date"].max():%Y-%m-%d}: '
    f'{len(factor_returns):,} trading dates; '
    f'{factor_returns.shape[1] - 1} factors'
)
factor_returns.tail()

In [ ]:
ax = create_correlation_heatmap(data=factor_returns)
plt.tight_layout()
plt.show()

Pass a source-module theme such as `value`, `quality`, `momentum`, or `investment` to focus the heatmap.

In [ ]:
THEME = None  # For example: 'investment'
if THEME is not None:
    ax = create_correlation_heatmap(data=factor_returns, theme=THEME)
    plt.tight_layout()
    plt.show()